In [2]:
#from google.colab import drive
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# ==============================================================================
# 1. MONTAR GOOGLE DRIVE
# ==============================================================================
#print("Montando Google Drive...")
#drive.mount('/content/drive')

Montando Google Drive...
Mounted at /content/drive


In [6]:
"""
================================================================================
SCRIPT DE AUDITORÍA DE DATOS DE CAMPO (CAMPAMENTO TAMUL)
Objetivo: Generar un reporte de inconsistencias para corrección manual.
================================================================================
"""

import pandas as pd
import numpy as np

def auditar_base_datos():
    print("=" * 80)
    print(" INICIANDO AUDITORÍA DE LA BASE DE DATOS ORIGINAL ")
    print("=" * 80)

    archivo_entrada = '/Users/antonio/Desktop/PT_Tortugas/Data/Data_Joined.csv'

    try:
        df = pd.read_csv(archivo_entrada)
        print(f"Archivo cargado exitosamente. Total de registros a auditar: {len(df)}")
    except FileNotFoundError:
        print(f"[ERROR] No se encontró el archivo '{archivo_entrada}'.")
        return

    # Lista para ir guardando los errores detectados
    reporte_errores = []

    # 1. CONTEO GENERAL POR AÑO
    print("\n[1] Resumen de Nidos por Año:")
    conteo_anual = df['year'].value_counts().sort_index()
    for anio, conteo in conteo_anual.items():
        print(f"    - {anio}: {conteo} nidos")

    # 2. PREPARACIÓN DE FECHAS (Para calcular duraciones)
    # Intentamos convertir las fechas. Si están mal escritas (ej. 32/14/2018), se volverán NaT (Not a Time)
    df['fecha_siembra_temp'] = pd.to_datetime(df['fecha_recolecta'] + ' ' + df['hora_siembra'], errors='coerce')
    df['fecha_emergencia_temp'] = pd.to_datetime(df['fecha_emergencia'] + ' 20:00:00', errors='coerce')

    # Calculamos los días brutos de incubación
    df['dias_incubacion_calc'] = (df['fecha_emergencia_temp'] - df['fecha_siembra_temp']).dt.total_seconds() / (3600.0 * 24.0)

    print("\n[2] Escaneando inconsistencias fila por fila...")

    # Iterar sobre cada fila para aplicar las reglas de auditoría
    for index, fila in df.iterrows():
        nido_id = f"Año: {fila['year']} | Legacy ID: {fila.get('nido_legacy', 'N/A')}"

        # REGLA 1: Fechas faltantes o con formato incorrecto
        if pd.isna(fila['fecha_siembra_temp']):
            reporte_errores.append({'Nido': nido_id, 'Columna': 'fecha_recolecta / hora_siembra', 'Error': 'Fecha u hora en blanco o formato inválido.'})
            continue # Si no hay fecha válida, no podemos calcular la duración

        if pd.isna(fila['fecha_emergencia_temp']):
            reporte_errores.append({'Nido': nido_id, 'Columna': 'fecha_emergencia', 'Error': 'Fecha de emergencia en blanco o inválida.'})
            continue

        # REGLA 2: Incubación Biológicamente Imposible (Zombis)
        dias = fila['dias_incubacion_calc']
        if dias > 75:
            reporte_errores.append({'Nido': nido_id, 'Columna': 'fecha_emergencia', 'Error': f'Incubación demasiado larga ({dias:.1f} días). Probable error de año.'})
        elif dias < 40:
            reporte_errores.append({'Nido': nido_id, 'Columna': 'fecha_emergencia', 'Error': f'Incubación demasiado corta ({dias:.1f} días).'})

        # REGLA 3: Huevos Sembrados Anómalos
        huevos = fila.get('huevos_sembrados', 0)
        if pd.isna(huevos) or huevos <= 0:
            reporte_errores.append({'Nido': nido_id, 'Columna': 'huevos_sembrados', 'Error': 'El nido tiene 0 huevos o está en blanco.'})
        elif huevos > 200:
            reporte_errores.append({'Nido': nido_id, 'Columna': 'huevos_sembrados', 'Error': f'Cantidad atípicamente alta ({huevos} huevos). Verificar si es correcto.'})

        # REGLA 4: Física de Eclosión (No pueden salir más de los que entraron)
        # Sumamos: crias vivas + muertas + huevos sin desarrollo (hsda) + huevos con desarrollo (hcda)
        crias_vivas = fila.get('crias_vivas', 0) if not pd.isna(fila.get('crias_vivas', 0)) else 0
        crias_muertas = fila.get('crias_muertas', 0) if not pd.isna(fila.get('crias_muertas', 0)) else 0
        hsda = fila.get('hsda', 0) if not pd.isna(fila.get('hsda', 0)) else 0
        hcda = fila.get('hcda', 0) if not pd.isna(fila.get('hcda', 0)) else 0

        total_hallazgos = crias_vivas + crias_muertas + hsda + hcda

        if total_hallazgos > huevos:
            reporte_errores.append({
                'Nido': nido_id,
                'Columna': 'Resultados de Eclosión',
                'Error': f'Inconsistencia matemática: Se sembraron {huevos} huevos, pero la suma de hallazgos da {total_hallazgos}.'
            })

        # REGLA 5: Especie Desconocida
        especie = str(fila.get('especie', '')).strip().lower()
        if especie not in ['cc', 'cm', 'ei']:
            reporte_errores.append({'Nido': nido_id, 'Columna': 'especie', 'Error': f"Código de especie desconocido ('{fila.get('especie', '')}'). Usar Cc, Cm o Ei."})

    # 3. EXPORTAR EL REPORTE
    print("\n" + "=" * 80)
    if len(reporte_errores) == 0:
        print(" ¡FELICIDADES! No se encontraron inconsistencias graves en la base de datos.")
    else:
        df_errores = pd.DataFrame(reporte_errores)
        archivo_salida = 'reporte_inconsistencias_alumnos.csv'
        df_errores.to_csv(archivo_salida, index=False, encoding='utf-8-sig')
        print(f" [ALERTA] Se detectaron {len(reporte_errores)} posibles errores.")
        print(f" Se ha generado el archivo '{archivo_salida}' para que los alumnos lo revisen.")
        print("\n Muestra de los primeros 5 errores encontrados:")
        print(df_errores.head())
    print("=" * 80)

if __name__ == "__main__":
    auditar_base_datos()

 INICIANDO AUDITORÍA DE LA BASE DE DATOS ORIGINAL 
Archivo cargado exitosamente. Total de registros a auditar: 4882

[1] Resumen de Nidos por Año:


KeyError: 'year'